## Imports

In [1]:
import math
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from transformers import get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import wandb

print('Imports done')

Imports done


## Wandb Login

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

print('Wandb succesfully logged in')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Wandb succesfully logged in


## Initialisation

In [3]:
WANDB_PROJECT = "22f3002631-t22026"
WANDB_ENTITY  = "22f3002631-iit-madras"
WANDB_GROUP   = "scratch"   # groups all folds together
LOG_MLM_RUN   = True                       # also log the MLM pretraining as its own run

TRAIN_CSV     = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_CSV      = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

OPTIONS = ["A", "B", "C", "D", "E"]
LETTER_TO_IDX = {l: i for i, l in enumerate(OPTIONS)}
PAD, UNK, CLS, SEP, MASK = 0, 1, 2, 3, 4
SPECIAL = {"[PAD]": PAD, "[UNK]": UNK, "[CLS]": CLS, "[SEP]": SEP, "[MASK]": MASK}
N_SPECIAL = 5
N_LEX = 3
FOLDS = 5
MAX_LEN = 256
D_MODEL = 256
NHEAD = 4
NLAYERS = 4
DIM_FF = 1024
DROPOUT = 0.3

MLM_EPOCHS = 40
MLM_LR = 5e-4
MLM_BATCH = 64

EPOCHS = 35
LR = 4e-4
WEIGHT_DECAY = 0.01
BATCH_SIZE = 32
WARMUP_FRAC = 0.1
PATIENCE = 7
MASK_PROB = 0.10
SEED = 42


torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device = {DEVICE}")

Device = cuda


## Functions

In [4]:
def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())

def compute_map3(scores, labels):
    ranked = np.argsort(-scores, axis=1)
    aps = []
    for i in range(len(labels)):
        ap = 0.0
        for rank in range(3):
            if ranked[i, rank] == labels[i]:
                ap = 1.0 / (rank + 1)
                break
        aps.append(ap)
    return np.mean(aps)

# build vocabularies for our model
def build_vocab(texts):
    c = Counter()
    for t in texts:
        c.update(tokenize(t))
    vocab = dict(SPECIAL)
    for tok, freq in c.most_common():
        vocab[tok] = len(vocab)
    return vocab

# segment prompts as 0 and options as 1
def encode_pair(prompt, option, vocab, MAX_LEN):
    p = [vocab.get(t, UNK) for t in tokenize(prompt)]
    o = [vocab.get(t, UNK) for t in tokenize(option)]
    ids = ([CLS] + p + [SEP] + o + [SEP])[:MAX_LEN]
    seg = ([0] * (len(p) + 2) + [1] * (len(o) + 1))[:MAX_LEN]
    return ids, seg

# jaccard similarity, option coverage, log of overlap count
def lexical_features(prompt, option):
    ptoks = set(tokenize(prompt))
    otoks = set(tokenize(option))
    inter = len(ptoks & otoks)
    union = len(ptoks | otoks) or 1
    # taking log so raw long text doesn't dominate
    return [inter / union, inter / (len(otoks) + 1), math.log1p(inter)]

## Shared Encoder

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, nhead=NHEAD, 
            nlayers=NLAYERS,dim_ff=DIM_FF,max_len=MAX_LEN, dropout=DROPOUT):
        super().__init__()

        # token, positional and segment embed
        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos = nn.Embedding(max_len, d_model)
        self.seg = nn.Embedding(2, d_model)
        self.drop = nn.Dropout(dropout)

        # single transform encoder layer
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,nhead=nhead,dim_feedforward=dim_ff,dropout=dropout,
            activation="gelu",batch_first=True,norm_first=True)

        # stacking 4 layers
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.d_model = d_model
        self.apply(self._init)   

    # custom weight initialization
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)
            with torch.no_grad():
                m.weight[PAD].zero_()

    # combine all 3 embeddings
    def forward(self, input_ids, seg_ids, pad_mask):
        S = input_ids.size(1)
        pos = torch.arange(S, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.tok(input_ids) + self.pos(pos) + self.seg(seg_ids))
        return self.enc(x, src_key_padding_mask=pad_mask)

## Phase 1 : MLM

In [6]:
class MLMModel(nn.Module):
    def __init__(self, encoder, vocab_size):
        super().__init__()
        self.encoder = encoder
        d = encoder.d_model
        self.transform = nn.Sequential(
                    nn.Linear(d, d),
                    nn.GELU(),
                    nn.LayerNorm(d)
                )
        self.decoder = nn.Linear(d, vocab_size, bias=True)
        self.decoder.weight = self.encoder.tok.weight  # weight tying

    def forward(self, input_ids, seg_ids, pad_mask):
        h = self.encoder(input_ids, seg_ids, pad_mask)
        return self.decoder(self.transform(h))


class MLMDataset(Dataset):      # Each (prompt, option) pair becomes one seq
    def __init__(self, df, vocab, MAX_LEN):
        self.seqs = []
        for _, r in df.iterrows():
            for opt in OPTIONS:
                ids, seg = encode_pair(r["prompt"], r[opt], vocab, MAX_LEN)
                self.seqs.append((ids, seg))

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, i):
        return self.seqs[i]


## MLM collate

In [7]:
def mlm_collate(vocab_size):
    def collate(batch):
        maxlen = max(len(ids) for ids, _ in batch)
        mask_rate=0.15      # BERT based
        N = len(batch)
        input_ids = np.full((N, maxlen), PAD, dtype=np.int64)
        seg_ids = np.zeros((N, maxlen), dtype=np.int64)
        pad_mask = np.ones((N, maxlen), dtype=bool)
        labels = np.full((N, maxlen), -100, dtype=np.int64)     # CE default ignore index
        for i, (ids, seg) in enumerate(batch):
            L = len(ids)
            input_ids[i, :L] = ids
            seg_ids[i, :L] = seg
            pad_mask[i, :L] = False
            for j in range(L):
                tid = ids[j]
                if tid < N_SPECIAL:   # never mask specials
                    continue
                if np.random.rand() < mask_rate:
                    labels[i, j] = tid
                    p = np.random.rand()
                    # mask 80%
                    if p < 0.8:
                        input_ids[i, j] = MASK
                    # substitute random words from vocabulary for 10%
                    elif p < 0.9:
                        input_ids[i, j] = np.random.randint(N_SPECIAL,vocab_size)
                    # else keep original token
        return (torch.from_numpy(input_ids), torch.from_numpy(seg_ids),
                torch.from_numpy(pad_mask), torch.from_numpy(labels))
    return collate

## MLM train

In [8]:
def pretrain_mlm(df, vocab):
    ds = MLMDataset(df, vocab, MAX_LEN)

    # creating dataloader with bert style masking
    loader = DataLoader(ds, batch_size=MLM_BATCH, shuffle=True,
                        collate_fn=mlm_collate(len(vocab)))
    encoder = Encoder(len(vocab), D_MODEL, NHEAD, 
                      NLAYERS,DIM_FF, MAX_LEN, DROPOUT).to(DEVICE)
    
    # MLM pred head over encoder
    model = MLMModel(encoder, len(vocab)).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=MLM_LR,weight_decay=WEIGHT_DECAY)
    steps = MLM_EPOCHS * len(loader)
    warmup = int(0.1 * steps)

    sched = get_cosine_schedule_with_warmup(optimizer=optim,
                                        num_warmup_steps=warmup,num_training_steps=steps)

    # for predicted mask tokens
    crit = nn.CrossEntropyLoss()
    print(f"MLM pretraining on {len(ds)} sequences for {MLM_EPOCHS} epochs")

    mlm_run = None
    if LOG_MLM_RUN:
        mlm_run = wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, group=WANDB_GROUP,
            name="mlm-pretrain", job_type="pretrain", reinit=True,
            config={
                "stage": "mlm_pretrain", "mlm_epochs": MLM_EPOCHS, "mlm_lr": MLM_LR,
                "mlm_batch": MLM_BATCH, "d_model": D_MODEL, "nhead": NHEAD,
                "nlayers": NLAYERS, "dim_ff": DIM_FF, "dropout": DROPOUT,
                "MAX_LEN": MAX_LEN, "vocab_size": len(vocab),
            },
            tags=["MLM"],
            settings=wandb.Settings(silent=True)
        )

    model.train()
    for ep in range(1, MLM_EPOCHS + 1):
        tot, correct, n_masked = 0.0, 0, 0
        for input_ids, seg_ids, pad_mask, labels in loader:
            input_ids = input_ids.to(DEVICE)
            seg_ids = seg_ids.to(DEVICE)
            pad_mask = pad_mask.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(input_ids, seg_ids, pad_mask)
            loss = crit(logits.view(-1, logits.size(-1)), labels.view(-1))

            # backprop and param update
            optim.zero_grad()
            loss.backward()

            # preventing grads from exploding
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            sched.step()
            tot += loss.item()
            sel = labels.view(-1) != -100

            if sel.any():
                pred = logits.view(-1, logits.size(-1)).argmax(-1)[sel]
                correct += (pred == labels.view(-1)[sel]).sum().item()
                n_masked += sel.sum().item()

        # avg epoch loss and masked token acc
        mlm_loss = tot / len(loader)
        mlm_acc  = correct / max(1, n_masked)
        if mlm_run is not None:
            wandb.log({"epoch": ep, "mlm/loss": mlm_loss,
                       "mlm/masked_token_acc": mlm_acc})
        if ep % 5 == 0 or ep == 1:
            print(f"  mlm epoch {ep:2d}  loss={mlm_loss:.4f}  "
                  f"masked-token acc={mlm_acc:.3f}")

    if mlm_run is not None:
        wandb.summary["mlm/final_loss"] = mlm_loss
        wandb.summary["mlm/final_masked_token_acc"] = mlm_acc
        wandb.finish()

    return {k: v.detach().cpu().clone()
            for k, v in encoder.state_dict().items()}

## MCQ model

In [9]:
class MCQModel(nn.Module):
    def __init__(self, encoder, dropout):
        super().__init__()

        # pretrained transformer
        self.encoder = encoder
        d = encoder.d_model
        self.norm = nn.LayerNorm(2 * d)

        # score for each opt
        self.head = nn.Sequential(
            nn.Linear(2 * d + N_LEX, d), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d, 1))

    def forward(self, input_ids, seg_ids, pad_mask, lex, B, nc):
        h = self.encoder(input_ids, seg_ids, pad_mask)
        cls = h[:, 0, :]
        real = (~pad_mask).unsqueeze(-1).float()

        # mean pooling of valid tokens
        mean = (h * real).sum(1) / real.sum(1)
        #combine cls and mean representation
        pooled = self.norm(torch.cat([cls, mean], dim=-1))
        feat = torch.cat([pooled, lex], dim=-1)
        return self.head(feat).squeeze(-1).view(B, nc)

class MCQDataset(Dataset):
    def __init__(self, df, vocab, MAX_LEN, has_label=True):
        self.rows, self.labels = [], []
        #encode every opt
        for _, r in df.iterrows():
            ch = []
            for opt in OPTIONS:
                ids, seg = encode_pair(r["prompt"], r[opt], vocab, MAX_LEN)
                ch.append((ids, seg, lexical_features(r["prompt"], r[opt])))

            # store all opt for each qsn
            self.rows.append(ch)
            self.labels.append(LETTER_TO_IDX[r["answer"].strip()] if has_label else -1)

    # no of mcq qsn
    def __len__(self):
        return len(self.rows)
    
    def __getitem__(self, i):
        return self.rows[i], self.labels[i]

## MCQ collate

In [10]:
def mcq_collate(batch):
    B = len(batch)
    nc = len(batch[0][0])

    # flattened seq
    f_inp, f_seg, f_lex = [], [], []
    for ch, _ in batch:
        for ids, seg, lex in ch:
            f_inp.append(ids)
            f_seg.append(seg)
            f_lex.append(lex)

    # finding longest seq for padding
    maxlen = max(len(x) for x in f_inp)
    N = len(f_inp)
    
    # create paddings
    input_ids = np.full((N, maxlen), PAD, dtype=np.int64)
    seg_ids = np.zeros((N, maxlen), dtype=np.int64)
    pad_mask = np.ones((N, maxlen), dtype=bool)
    
    # for i, (ids, seg) in enumerate(zip(f_inp, f_seg)):
    #     L = len(ids)
    #     input_ids[i, :L] = ids
    #     seg_ids[i, :L] = seg
    #     pad_mask[i, :L] = False
    
    # filling with actual values
    for i in range(len(f_inp)):
        L = len(f_inp[i])
        input_ids[i, :L] = f_inp[i]
        seg_ids[i, :L] = f_seg[i]
        pad_mask[i, :L] = False

    return (torch.from_numpy(input_ids), torch.from_numpy(seg_ids),
            torch.from_numpy(pad_mask),
            torch.from_numpy(np.asarray(f_lex, dtype=np.float32)),
            torch.from_numpy(np.asarray([lb for _, lb in batch], dtype=np.int64)),
            B, nc)

## EMA

In [11]:
class EMA:
    def __init__(self, model):
        self.decay = 0.999    # decay val for preserving old weights
        # save init copies of params
        self.shadow = {k: v.detach().clone()
                       for k, v in model.state_dict().items()}

    def update(self, model):
        # EMA update rule 
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(),alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

# label-less masking
def mask_augment(input_ids, prob):
    rand = torch.rand_like(input_ids, dtype=torch.float)
    do = (rand < prob) & (input_ids >= N_SPECIAL)
    return torch.where(do, torch.full_like(input_ids, MASK), input_ids)


## MCQ training

In [12]:
def mcq_epoch(model, loader, optim=None, sched=None, ema=None, MASK_PROB=0.0):
    train = optim is not None
    model.train(train)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    tot, sc, lb = 0.0, [], []
    
    for input_ids, seg_ids, pad_mask, lex, labels, B, nc in loader:
        # random token mask augment
        if train:
            input_ids = mask_augment(input_ids, MASK_PROB)
            
        input_ids = input_ids.to(DEVICE)
        seg_ids = seg_ids.to(DEVICE)
        pad_mask = pad_mask.to(DEVICE)
        lex = lex.to(DEVICE)
        labels = labels.to(DEVICE)
        
        with torch.set_grad_enabled(train):
            logits = model(input_ids, seg_ids, pad_mask, lex, B, nc)
            # mcq classification loss
            loss = crit(logits, labels)
            if train:
                optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
                if sched:
                    sched.step()
                if ema:
                    ema.update(model)
                    
        tot += loss.item() * B
        # store preds
        sc.append(logits.detach().cpu().numpy())
        lb.append(labels.cpu().numpy())

    # combine batches
    sc = np.concatenate(sc)
    lb = np.concatenate(lb)

    return (tot / len(loader.dataset), 
            float((sc.argmax(1) == lb).mean()),compute_map3(sc, lb))


## Helper functions

In [13]:
def predict_probs(model, loader):
    model.eval()
    probs = []

    with torch.no_grad():
        for input_ids, seg_ids, pad_mask, lex, _, B, nc in loader:
            # option scores from model
            logits = model(
                input_ids.to(DEVICE),seg_ids.to(DEVICE),
                pad_mask.to(DEVICE),lex.to(DEVICE),B,nc)
            
            # score into probs
            probs.append(F.softmax(logits, dim=1).cpu().numpy())

    return np.concatenate(probs, axis=0)


def eval_with_ema(model, ema, loader):
    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    # EMA smoothed weights for eval
    model.load_state_dict(ema.shadow)
    # eval using ema params
    res = mcq_epoch(model, loader)
    state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(backup)
    return res, state

## Main

In [14]:
import os
os.environ["WANDB_SILENT"] = "true" 

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# build vocab
texts = []
for df in (train_df, test_df):
    for col in ["prompt"] + OPTIONS:
        texts.extend(df[col].astype(str).tolist())
vocab = build_vocab(texts)
print(f"Vocabulary size = {len(vocab)}")

# combine and pretrain using MLM
mlm_df = pd.concat([train_df, test_df], ignore_index=True)
pretrained = pretrain_mlm(mlm_df, vocab)

# init pretrained encoder
def fresh_encoder():
    e = Encoder(len(vocab), D_MODEL, NHEAD, NLAYERS,DIM_FF, MAX_LEN, DROPOUT)
    e.load_state_dict(pretrained)
    return e

# PHASE 2 

y = train_df["answer"].map(LETTER_TO_IDX).values
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True,random_state=SEED)
test_loader = DataLoader(
    MCQDataset(test_df, vocab, MAX_LEN, has_label=False),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=mcq_collate)

# store preds
test_probs = np.zeros((len(test_df), 5), dtype=np.float64)
oof_probs = np.zeros((len(train_df), 5), dtype=np.float64)  # for stacking
fmp3, facc, ff1 = [], [], []

# training fold
for fold, (tr_idx, vl_idx) in enumerate(skf.split(train_df, y), 1):
    tr_loader = DataLoader(
        MCQDataset(train_df.iloc[tr_idx].reset_index(drop=True), vocab,MAX_LEN),
        batch_size=BATCH_SIZE, shuffle=True, collate_fn=mcq_collate)
    vl_ds = MCQDataset(train_df.iloc[vl_idx].reset_index(drop=True), vocab,MAX_LEN)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE,shuffle=False, collate_fn=mcq_collate)

    model = MCQModel(fresh_encoder(), DROPOUT).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LR,weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * len(tr_loader)
    warmup = int(WARMUP_FRAC * steps)

    sched = get_cosine_schedule_with_warmup(
        optimizer=optim,
        num_warmup_steps=warmup,
        num_training_steps=steps,
    )
    ema = EMA(model)

    # W&B run for this fold 
    run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY, group=WANDB_GROUP,
        name=f"fold-{fold}", job_type="train", reinit=True,
        config={
            "stage": "mcq_finetune", "fold": fold, "n_folds": FOLDS,
            "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY,
            "batch_size": BATCH_SIZE, "warmup_frac": WARMUP_FRAC,
            "patience": PATIENCE, "MASK_PROB": MASK_PROB, "label_smoothing": 0.1,
            "ema_decay": ema.decay, "d_model": D_MODEL, "nhead": NHEAD,
            "nlayers": NLAYERS, "dim_ff": DIM_FF, "dropout": DROPOUT,
            "max_len": MAX_LEN, "vocab_size": len(vocab), "seed": SEED,
            "n_train": len(tr_idx), "n_val": len(vl_idx),
        },
        tags=["EMA"],
        settings=wandb.Settings(silent=True)
    )

    # track best model
    best_mp3, best_state, bad = -1.0, None, 0

    # training loop
    for ep in range(1, EPOCHS + 1):
        tr_loss, tr_acc, tr_mp3 = mcq_epoch(model, tr_loader, optim, sched, ema, MASK_PROB)
        (vl_loss, vl_acc, vmp3), st = eval_with_ema(model, ema, vl_loader)

        wandb.log({
            "epoch": ep,
            "train/loss": tr_loss, "train/accuracy": tr_acc, "train/map@3": tr_mp3,
            "val/loss": vl_loss, "val/accuracy": vl_acc, "val/map@3": vmp3,
            "lr": sched.get_last_lr()[0],
        })

        if vmp3 > best_mp3:
            best_mp3, best_state, bad = vmp3, st, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[fold {fold}/{FOLDS}] early stop at epoch {ep}")
                break
    print(f"[fold {fold}/{FOLDS}] best val map@3={best_mp3:.4f}")

    # load best weight and eval
    model.load_state_dict(best_state)
    _, vacc, vmp3 = mcq_epoch(model, vl_loader)
    vprobs = predict_probs(model, vl_loader)
    oof_probs[vl_idx] = vprobs          # out-of-fold, these rows weren't trained on

    # compute f1
    vf1 = f1_score(np.array(vl_ds.labels), vprobs.argmax(1), average="macro")
    ff1.append(vf1)
    fmp3.append(vmp3)
    facc.append(vacc)
    test_probs += predict_probs(model, test_loader)

    # best-epoch fold metrics into run summary
    wandb.summary["best/val_accuracy"] = vacc
    wandb.summary["best/val_f1_macro"] = vf1
    wandb.summary["best/val_map@3"]    = vmp3
    wandb.log({"final/val_accuracy": vacc, "final/val_f1_macro": vf1,
               "final/val_map@3": vmp3})
    wandb.finish()

# avg pred
test_probs /= FOLDS

# avg cv scores
cv_mp3 = float(np.mean(fmp3))
cv_acc = float(np.mean(facc))
cv_f1 = float(np.mean(ff1))

print(f"\n[CV ] map@3={cv_mp3:.4f} acc={cv_acc:.4f} f1_macro={cv_f1:.4f} "f"(mean over {FOLDS} FOLDS)")

# CV summary run: mean/std across FOLDS + OOF
oof_map3 = compute_map3(oof_probs, y)
oof_acc  = float((oof_probs.argmax(1) == y).mean())
oof_f1   = f1_score(y, oof_probs.argmax(1), average="macro")

wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group=WANDB_GROUP,
           name="cv-summary", job_type="eval", reinit=True,
           config={"n_folds": FOLDS, "seed": SEED}, tags=["summary"],
          settings=wandb.Settings(silent=True))

wandb.summary.update({
    "cv/accuracy_mean": cv_acc, "cv/accuracy_std": float(np.std(facc)),
    "cv/f1_macro_mean": cv_f1,  "cv/f1_macro_std": float(np.std(ff1)),
    "cv/map@3_mean":    cv_mp3, "cv/map@3_std":    float(np.std(fmp3)),
    "oof/accuracy": oof_acc, "oof/f1_macro": oof_f1, "oof/map@3": oof_map3,
})

wandb.finish()


order = np.argsort(-test_probs, axis=1)
preds = [" ".join(OPTIONS[j] for j in row[:3]) for row in order]
pd.DataFrame({"ID": test_df["id"], "Prediction": preds}).to_csv('submission.csv', index=False)
print(f"[done] wrote {len(preds)} predictions -> submission.csv")

Vocabulary size = 3007


/tmp/ipykernel_23/2310811315.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)


MLM pretraining on 12500 sequences for 40 epochs


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


  mlm epoch  1  loss=7.0365  masked-token acc=0.091
  mlm epoch  5  loss=4.7191  masked-token acc=0.218
  mlm epoch 10  loss=3.0407  masked-token acc=0.367
  mlm epoch 15  loss=2.0372  masked-token acc=0.531
  mlm epoch 20  loss=1.4763  masked-token acc=0.649
  mlm epoch 25  loss=1.1667  masked-token acc=0.716
  mlm epoch 30  loss=0.9898  masked-token acc=0.756
  mlm epoch 35  loss=0.9310  masked-token acc=0.772
  mlm epoch 40  loss=0.9191  masked-token acc=0.775


...

[fold 3/5] best val map@3=0.9975


/tmp/ipykernel_23/2310811315.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)


[fold 4/5] early stop at epoch 35
[fold 4/5] best val map@3=0.9975


/tmp/ipykernel_23/2310811315.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)


[fold 5/5] early stop at epoch 31
[fold 5/5] best val map@3=0.9938

[CV ] map@3=0.9949 acc=0.9900 f1_macro=0.9901 (mean over 5 FOLDS)
[done] wrote 500 predictions -> submission.csv
